# Tutorial 1 — Getting started

**Goal:** run a complete dual-modality simulation in a few lines and learn what each
stage of the pipeline does.

**Time:** ~2 minutes on a laptop CPU. No GPU needed. Every tutorial runs on the
NumPy/scikit-image backends; if the ASTRA toolbox is installed, the same code runs on the GPU.

**Before you start:** install the package from the repository root:
```bash
pip install -e ".[notebooks]"
```

### The big idea
X-rays are attenuated mostly by **heavy elements** (metals); thermal neutrons mostly by
**light, hydrogen-rich** materials (water, polymers). If we reconstruct the *same* sample with
both, every voxel gets two numbers — its X-ray attenuation $\mu_x$ and its neutron attenuation
$\mu_n$. Plotting all voxels in the $(\mu_x, \mu_n)$ plane gives the **bimodal histogram**:
each material becomes a separate cluster, even materials that are indistinguishable in one modality.

DIANA simulates the whole chain so we can study **how acquisition imperfections distort
that histogram**:

```
phantom ─► projection ─► artifacts ─► reconstruction ─► bimodal histogram ─► metrics
```

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import neutron_xray_sim as nxs

print("DIANA / neutron_xray_sim", nxs.__version__)

## 1. One-line simulation

`DualModalitySimulation` wraps the full pipeline. We pick a **preset phantom**, a grid size `N`
(voxels per side) and the number of projection angles. Small values keep things fast.

In [ ]:
sim = nxs.DualModalitySimulation(
    preset="composite",   # HDPE cylinder with water, iron, titanium inclusions
    N=48,                 # 48³ voxels
    n_angles=90,          # projections over 180°
    verbose=False,
)
print(sim.phantom)

## 2. A clean run and a realistic run

`ArtifactConfig` switches acquisition artifacts on and off. `clean()` disables everything
(except beam hardening, which is intrinsic to a polychromatic X-ray tube); `realistic()` turns
on noise, scatter, detector blur, ring artifacts and inter-modality misalignment.

In [ ]:
clean = sim.run(nxs.ArtifactConfig.clean(), tag="clean")
real  = sim.run(nxs.ArtifactConfig.realistic(), tag="realistic", ref_result=clean)
print(real.summary())

Each run returns a `SimulationResult` holding everything that was computed:

In [ ]:
print("reconstructed X-ray volume :", real.vol_xray.shape, "cm⁻¹")
print("reconstructed neutron volume:", real.vol_neutron.shape, "cm⁻¹")
print("sinogram (angles, slices, detector):", real.xray_sino["sino_lam"].shape)
print("histogram bins:", real.histogram.H.shape)

## 3. Look at the reconstructions

The same slice looks completely different in the two modalities: the iron rod dominates the
X-ray image, while the hydrogen-rich HDPE matrix dominates the neutron image.

In [ ]:
fig = clean.plot_slices()

## 4. The bimodal histogram

Now compare the two runs' histograms. In the clean run each material is a compact blob; the
artifacts smear, shift and stretch them.

In [ ]:
fig = sim.comparison_grid([clean, real], ncols=2)

The white clusters can be compared with the *exact* material positions from the phantom —
the ground truth that only a simulation gives you:

In [ ]:
fig = nxs.plot_ground_truth_comparison(sim.phantom, clean.histogram,
                                      title_recon="Clean reconstruction")

## 5. Put a number on it

Because the simulation knows which voxels truly belong to each material, we can ask where
those voxels ended up in the histogram. The *label-anchored* metrics report:

* **ε_k** — per material, the distance between where its voxels landed and its true position (cm⁻¹)
* **CE** — the mean of ε_k (lower is better)
* **DB** — Davies–Bouldin index: how well the clusters are separated (lower is better)

In [ ]:
for r in (clean, real):
    table, _ = nxs.compute_histogram_metrics_morphology_aware(
        sim.phantom, r.histogram, r.vol_xray, r.vol_neutron)
    eps = ", ".join(f"{k} {v:.2f}" for k, v in table.per_cluster["eps_k"].items())
    print(f"{r.tag:>10}:  CE = {table.scalars['CE']:.3f} cm⁻¹  DB = {table.scalars['DB']:.3f}   (ε_k: {eps})")

Two things to notice:

* the realistic run is worse on both metrics, as expected;
* even the **clean** run puts iron far from its "true" position. The ground truth uses μ at
  80 keV, but the polychromatic beam is *hardened* differently along each ray, so dense metals
  reconstruct at a different effective energy. This is beam hardening, and it is present in
  every lab X-ray scan unless corrected.

Tutorial 5 introduces the other metric families, including GMM-based ones that work
without ground-truth labels.

## Where next?

| Tutorial | Topic |
|---|---|
| 2 | Materials and phantoms — build your own sample |
| 3 | Forward projection and artifacts — what each artifact does to the data |
| 4 | Reconstruction — algorithms and their fidelity |
| 5 | Bimodal-histogram analysis — GMMs, segmentation and metrics |
| 6 | Artifact studies — sweeps, surveys and caching results to disk |
| 7 | Neutron beams — cold and polychromatic spectra |

**Exercise:** change `preset` to `"battery"` or `"bone_implant"` and rerun the notebook.
Which materials become separable only thanks to the second modality?